![image](car.jpeg)

**Car-ing is sharing**, an auto dealership company for car sales and rental, is taking their services to the next level thanks to **Large Language Models (LLMs)**.

As their newly recruited AI and NLP developer, you've been asked to prototype a chatbot app with multiple functionalities that not only assist customers but also provide support to human agents in the company.

The solution should receive textual prompts and use a variety of pre-trained Hugging Face LLMs to respond to a series of tasks, e.g. classifying the sentiment in a car’s text review, answering a customer question, summarizing or translating text, etc.


In [59]:
# Import necessary packages
import pandas as pd
import torch

from transformers import logging
logging.set_verbosity(logging.WARNING)

In [60]:
from transformers import pipeline
import evaluate
import pandas as pd

# Load data correctly
df = pd.read_csv('data/car_reviews.csv', sep=';')

# Load the pipeline
sentiment_analyzer = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

# 1. Store the direct model outputs here as requested (list of dictionaries)
predicted_labels = sentiment_analyzer(df['Review'].iloc[:5].tolist(), truncation=True)

# 2. Map predictions and references to integer binary labels {0, 1}
predictions = [1 if pred['label'] == "POSITIVE" else 0 for pred in predicted_labels]
references = [1 if label == "POSITIVE" else 0 for label in df['Class'].iloc[:5]]

# Load metrics
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

# 3. Compute the raw dictionary evaluations
raw_accuracy = accuracy_metric.compute(predictions=predictions, references=references)
raw_f1 = f1_metric.compute(predictions=predictions, references=references)

accuracy_result = raw_accuracy['accuracy']
f1_result = raw_f1['f1']

# Verify the formats are correct
print(f"Accuracy: {accuracy_result} (Type: {type(accuracy_result)})")
print(f"F1 Score: {f1_result} (Type: {type(f1_result)})")

Device set to use cpu


Accuracy: 0.8 (Type: <class 'float'>)
F1 Score: 0.8571428571428571 (Type: <class 'float'>)


In [61]:
from transformers import pipeline
import evaluate

# Load translation pipeline
translator = pipeline("translation", model="Helsinki-NLP/opus-mt-en-es")

# Isolate first review text
first_review = df['Review'].iloc[0]

# Translate specifying the max_length parameter to get the first two sentences naturally
translated_output = translator(first_review, max_length=27)
translated_review = translated_output[0]['translation_text']

# Preprocess reference translations
with open("data/reference_translations.txt", 'r', encoding='utf-8') as file:
    lines = file.readlines()
references_clean = [line.strip() for line in lines if line.strip()]

# Load and compute BLEU metric
bleu_metric = evaluate.load("bleu")

bleu_score = bleu_metric.compute(predictions=[translated_review], references=[references_clean])

# Verify format
print(bleu_score)

Device set to use cpu
Your input_length: 365 is bigger than 0.9 * max_length: 27. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)


{'bleu': 0.6022774485691839, 'precisions': [0.9090909090909091, 0.7142857142857143, 0.55, 0.3684210526315789], 'brevity_penalty': 1.0, 'length_ratio': 1.0476190476190477, 'translation_length': 22, 'reference_length': 21}


In [62]:
from transformers import pipeline

# Formulate input variables precisely
question = "What did he like about the brand?"
context = df['Review'].iloc[1]

# Load extractive QA pipeline
qa_pipeline = pipeline("question-answering", model="deepset/minilm-uncased-squad2")

# Get results
qa_result = qa_pipeline(question=question, context=context)
answer = qa_result['answer']

Some weights of the model checkpoint at deepset/minilm-uncased-squad2 were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


In [63]:
from transformers import pipeline

# Load standard summarization pipeline
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

# Isolate the last review text
last_review = df['Review'].iloc[-1]

# Generate summary between 50 and 55 tokens long
summary_output = summarizer(
    last_review,
    max_length=55,
    min_length=50,
    do_sample=False
)

# Store text in summarized_text variable
summarized_text = summary_output[0]['summary_text']

Device set to use cpu
